# Lesson 3.3 — Integration Exercise: Pose to Plan, Wired End to End

Wire `perceive → understand → to_configuration → plan_reference` in one run, emit a trace, and verify the three-link contract chain. Then break a link and watch the chain stop at the right stage.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, plan_reference, fk_xy, P2, T2, REACH_MAX)
checks = []
def run_seam(world, q_start, rng_seed=7):
    trace = []
    dets = model_perception(world, rng=np.random.default_rng(rng_seed))
    trace.append(('Perceive', 'world -> %d detections' % len(dets)))
    _, tgt = understand(dets, world)
    trace.append(('Understand', 'target=%s' % (tgt['id'] if tgt else None)))
    if tgt is None: return trace, None
    q_goal = to_configuration(tgt)
    if q_goal is None:
        trace.append(('IK', 'None (unreachable)')); return trace, None
    fk_err = float(np.linalg.norm(fk_xy(P2,T2,q_goal) - tgt['xy']))
    trace.append(('IK', 'q_goal ok, FK err=%.1e' % fk_err))
    layer = plan_reference(q_start, q_goal, rng=np.random.default_rng(0))
    end_err = float(np.linalg.norm(fk_xy(P2,T2,layer['reference'](layer['duration'])[0]) - tgt['xy']))
    trace.append(('Plan', 'validated=%s, endpoint err=%.1e' % (layer['validated'], end_err)))
    return trace, dict(target=tgt, q_goal=q_goal, layer=layer, fk_err=fk_err, end_err=end_err)


In [2]:
world = GreenhouseWorld.demo_row(n=6, seed=1)
trace, res = run_seam(world, world.q.copy())
for s, msg in trace: print(f'{s:11s} | {msg}')
# three-link contract chain
checks.append(res is not None)
checks.append(res['target']['ripe'] and res['target']['reachable'])  # link 1
checks.append(res['fk_err'] < 1e-9)                                  # link 2
checks.append(res['layer']['validated'] and res['end_err'] < 1e-3)   # link 3


Perceive    | world -> 6 detections
Understand  | target=F3
IK          | q_goal ok, FK err=6.2e-17
Plan        | validated=True, endpoint err=1.2e-15


### Break link 2: an out-of-reach target stops the chain at IK

In [3]:
broken = GreenhouseWorld(fruit=[Fruit('FX', REACH_MAX+0.3, 0.0, ripe=True)],
                          q=np.array([0.3,0.6]))
trace2, res2 = run_seam(broken, broken.q.copy())
for s, msg in trace2: print(f'{s:11s} | {msg}')
checks.append(res2 is None and trace2[-1][0] in ('Understand','IK'))  # stopped before Plan
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


Perceive    | world -> 1 detections
Understand  | target=None
5/5 checks passed.
All checks passed.
